In [9]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Find project directory
project_path = Path.cwd()

if project_path.name == "notebooks":
    project_path = project_path.parent

sys.path.insert(0, str(project_path))

# Import functions and class
from src.loader import load_dataset
from src.preprocessing_class import POSPreprocessor
from src.validation import validate_data

In [10]:
#This is to verify the lenght of the vocabulary, to set the vectorizer
unique_words = set(
    word
    for sentence in train_sentences
    for word in sentence
)

print("Unique words in training:", len(unique_words))

Unique words in training: 19674


In [11]:
# Load datasets
train_sentences, train_tags = load_dataset("en_ewt-ud-train.conllu")
dev_sentences, dev_tags = load_dataset("en_ewt-ud-dev.conllu")
test_sentences, test_tags = load_dataset("en_ewt-ud-test.conllu")

print("Training:", len(train_sentences))
print("Validation:", len(dev_sentences))
print("Test:", len(test_sentences))

Training: 12544
Validation: 2001
Test: 2077


In [12]:
# Convert words to numerical sequences
X_train, X_dev, X_test, vectorizer, max_length = preprocess_text(
    train_sentences,
    dev_sentences,
    test_sentences
)

print("Vocabulary size:", len(vectorizer.get_vocabulary()))
print("Max length:", max_length)

Vocabulary size: 19676
Max length: 159


In [13]:
# Convert POS tags to numerical IDs
y_train, y_dev, y_test, tag2id, id2tag = preprocess_tags(
    train_tags,
    dev_tags,
    test_tags,
    max_length
)

print("POS categories:", tag2id)

POS categories: {'ADJ': 0, 'ADP': 1, 'ADV': 2, 'AUX': 3, 'CCONJ': 4, 'DET': 5, 'INTJ': 6, 'NOUN': 7, 'NUM': 8, 'PART': 9, 'PRON': 10, 'PROPN': 11, 'PUNCT': 12, 'SCONJ': 13, 'SYM': 14, 'VERB': 15, 'X': 16}


In [14]:
# Check that input and output shapes match
assert X_train.shape == y_train.shape
assert X_dev.shape == y_dev.shape
assert X_test.shape == y_test.shape

# Check original sentence lengths
for sentences, tags in [
    (train_sentences, train_tags),
    (dev_sentences, dev_tags),
    (test_sentences, test_tags)
]:
    assert all(
        len(words) == len(labels)
        for words, labels in zip(sentences, tags)
    )

# Create masks for padding
w_train = (X_train != 0).astype("float32")
w_dev = (X_dev != 0).astype("float32")
w_test = (X_test != 0).astype("float32")

# Check number of real tokens
assert w_train.sum() == sum(len(s) for s in train_sentences)
assert w_dev.sum() == sum(len(s) for s in dev_sentences)
assert w_test.sum() == sum(len(s) for s in test_sentences)

print("All validations passed!")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_dev:", X_dev.shape)
print("y_dev:", y_dev.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

All validations passed!
X_train: (12544, 159)
y_train: (12544, 159)
X_dev: (2001, 159)
y_dev: (2001, 159)
X_test: (2077, 159)
y_test: (2077, 159)


In [15]:
# Select first sentence
sentence = train_sentences[0]
tags = train_tags[0]

# Display input-output correspondence
df = pd.DataFrame({
    "Word": sentence,
    "Word ID": X_train[0][:len(sentence)],
    "POS Tag": tags,
    "Tag ID": y_train[0][:len(tags)]
})

display(df)

,Word,Word ID,POS Tag,Tag ID
0,Al,263,PROPN,11
1,-,16,PUNCT,12
2,Zaman,6154,PROPN,11
3,:,45,PUNCT,12
4,American,293,ADJ,0
5,forces,709,NOUN,7
6,killed,1150,VERB,15
7,Shaikh,4859,PROPN,11
8,Abdullah,18887,PROPN,11
9,al,592,PROPN,11
